In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np

# 모델 관련
from sklearn.model_selection import StratifiedKFold, GridSearchCV, KFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

# 언더샘플링
from imblearn.under_sampling import RandomUnderSampler

# 부스터 모델
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 딥러닝 (CNN)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# -------------------------------
# 0. 데이터 불러오기
X = pd.read_csv("X_mm.csv")     # MinMaxScaler 버전 사용 예시
y = pd.read_csv("y.csv").squeeze()

# y 값 0/1로 변환 (중요!)
if set(y.unique()) == {1, 2}:
    y = y.map({1:0, 2:1}).astype(int)

print("로드 완료:", X.shape, y.shape, "Unique y:", y.unique())
# -------------------------------
# 1. 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba, fold):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)
    return {
        "Fold": fold,
        "Algorithm": name,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    }

# -------------------------------
# 2. 파라미터 그리드
param_grids = {
    "Logistic Regression": {'C': [0.1, 1, 10]},
    "Random Forest": {'n_estimators': [100, 200], 'max_depth': [5, 10]},
    "XGBoost": {'n_estimators': [100], 'max_depth': [3, 5]},
    "LightGBM": {'n_estimators': [100], 'num_leaves': [31, 50]},
    "CatBoost": {'depth': [4, 6], 'iterations': [100], 'learning_rate': [0.1]},
    "MLP": {'hidden_layer_sizes': [(64,), (64, 32)], 'alpha': [0.0001, 0.001]}
}

# -------------------------------
# 3. 기본 모델
base_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
    "MLP": MLPClassifier(max_iter=300, random_state=42)
}

# -------------------------------
# 4. 학습 + 평가 (5-Fold)
results = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    # 스케일링
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_valid_scaled = scaler.transform(X_valid)

    # 언더샘플링
    rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
    X_res, y_res = rus.fit_resample(X_train_scaled, y_train)

    # 전통 ML 모델
    for name, model in base_models.items():
        clf = GridSearchCV(model, param_grids[name], cv=3, scoring='f1', n_jobs=-1)
        clf.fit(X_res, y_res)
        best_model = clf.best_estimator_
        y_pred = best_model.predict(X_valid_scaled)
        y_proba = best_model.predict_proba(X_valid_scaled)[:, 1]
        results.append(evaluate_model(name, y_valid, y_pred, y_proba, fold))

    # CNN
    cnn = Sequential([
        Dense(64, activation='relu', input_shape=(X_res.shape[1],)),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    cnn.fit(X_res, y_res, epochs=10, batch_size=32, verbose=0)

    y_pred_cnn = cnn.predict(X_valid_scaled).flatten()
    y_pred_cnn_label = (y_pred_cnn >= 0.5).astype(int)
    results.append(evaluate_model("CNN", y_valid, y_pred_cnn_label, y_pred_cnn, fold))

# -------------------------------
# 5. KNN 평가 (별도)
X_scaled_all = MinMaxScaler().fit_transform(X)
X_res_all, y_res_all = RandomUnderSampler(sampling_strategy=0.5, random_state=42).fit_resample(X_scaled_all, y)

knn = KNeighborsClassifier(n_neighbors=5)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred_knn = cross_val_predict(knn, X_res_all, y_res_all, cv=kf, method='predict')
y_proba_knn = cross_val_predict(knn, X_res_all, y_res_all, cv=kf, method='predict_proba')[:, 1]
results.append(evaluate_model("KNN", y_res_all, y_pred_knn, y_proba_knn, fold='All'))

# -------------------------------
# 6. 결과 정리
result_df = pd.DataFrame(results)

summary = result_df.groupby("Algorithm")[[ 
    "Precision", "Recall", "Specificity", "F1-score", 
    "Accuracy", "Balanced Accuracy", "AUC"
]].mean().round(4).sort_values("AUC", ascending=False)

# -------------------------------
# 7. 결과 저장
result_df.to_csv("all_fold_results.csv", index=False, encoding="utf-8-sig")
summary.to_csv("model_summary.csv", encoding="utf-8-sig")

print("저장 완료")
print(" - all_fold_results.csv : Fold별 전체 성능 기록")
print(" - model_summary.csv : 모델별 평균 성능 요약")

# -------------------------------
# 8. 결과 출력
print(" 전체 Fold 결과:")
print(result_df.head())

print("\n 모델별 평균 성능 요약:")
print(summary)

In [ ]:
## 메모리 해결 버전

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np

# 모델 관련
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, KFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

# 언더샘플링
from imblearn.under_sampling import RandomUnderSampler

# 부스터 모델
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 딥러닝 (CNN)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# -------------------------------
# 0. 데이터 불러오기
X = pd.read_csv("X_mm.csv") ##### 파일 원하는걸루 바꾸세요.
y = pd.read_csv("y.csv").squeeze()

# y 값 확인 및 변환 (혹시 모를 [1,2] → [0,1])
if set(y.unique()) == {1, 2}:
    y = y.map({1:0, 2:1}).astype(int)

print(" 데이터 로드 완료:", X.shape, y.shape, "Unique y:", y.unique())

# -------------------------------
# 1. 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba, fold):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)
    return {
        "Fold": fold,
        "Algorithm": name,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    }

# -------------------------------
# 2. 파라미터 후보 (최소화)
param_grids = {
    "Logistic Regression": {'C': [0.1, 1, 10]},
    "Random Forest": {'n_estimators': [100, 200], 'max_depth': [5, 10]},
    "XGBoost": {'n_estimators': [100], 'max_depth': [3, 5]},
    "LightGBM": {'n_estimators': [100], 'num_leaves': [31, 50]},
    "CatBoost": {'depth': [4, 6], 'iterations': [100], 'learning_rate': [0.1]},
    "MLP": {'hidden_layer_sizes': [(64,), (64, 32)], 'alpha': [0.0001, 0.001]}
}

# -------------------------------
# 3. 기본 모델
base_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
    "MLP": MLPClassifier(max_iter=300, random_state=42)
}

# -------------------------------
# 4. 학습 + 평가 (5-Fold, 메모리 최적화)
results = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    # 스케일링
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
    X_valid_scaled = scaler.transform(X_valid).astype(np.float32)

    # 언더샘플링
    rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
    X_res, y_res = rus.fit_resample(X_train_scaled, y_train)

    # 메모리 절약을 위해 float32 변환
    X_res = X_res.astype(np.float32)

    # 전통 ML 모델 (RandomizedSearchCV + n_jobs=2)
    for name, model in base_models.items():
        clf = RandomizedSearchCV(
            model,
            param_distributions=param_grids[name],
            n_iter=2, cv=3, scoring='f1',
            n_jobs=2, random_state=42
        )
        clf.fit(X_res, y_res)
        best_model = clf.best_estimator_
        y_pred = best_model.predict(X_valid_scaled)
        y_proba = best_model.predict_proba(X_valid_scaled)[:, 1]
        results.append(evaluate_model(name, y_valid, y_pred, y_proba, fold))

    # CNN
    cnn = Sequential([
        Dense(64, activation='relu', input_shape=(X_res.shape[1],)),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    cnn.fit(X_res, y_res, epochs=5, batch_size=32, verbose=0)  # epochs 줄임

    y_pred_cnn = cnn.predict(X_valid_scaled).flatten()
    y_pred_cnn_label = (y_pred_cnn >= 0.5).astype(int)
    results.append(evaluate_model("CNN", y_valid, y_pred_cnn_label, y_pred_cnn, fold))

# -------------------------------
# 5. KNN 평가 (별도)
X_scaled_all = MinMaxScaler().fit_transform(X).astype(np.float32)
X_res_all, y_res_all = RandomUnderSampler(sampling_strategy=0.5, random_state=42).fit_resample(X_scaled_all, y)

knn = KNeighborsClassifier(n_neighbors=5)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred_knn = cross_val_predict(knn, X_res_all, y_res_all, cv=kf, method='predict')
y_proba_knn = cross_val_predict(knn, X_res_all, y_res_all, cv=kf, method='predict_proba')[:, 1]
results.append(evaluate_model("KNN", y_res_all, y_pred_knn, y_proba_knn, fold='All'))

# -------------------------------
# 6. 결과 정리 + 저장
result_df = pd.DataFrame(results)

summary = result_df.groupby("Algorithm")[[ 
    "Precision", "Recall", "Specificity", "F1-score", 
    "Accuracy", "Balanced Accuracy", "AUC"
]].mean().round(4).sort_values("AUC", ascending=False)

result_df.to_csv("all_fold_results.csv", index=False, encoding="utf-8-sig")
summary.to_csv("model_summary.csv", encoding="utf-8-sig")

print("저장 완료")
print(" - all_fold_results.csv : Fold별 전체 성능 기록")
print(" - model_summary.csv : 모델별 평균 성능 요약")

✅ 데이터 로드 완료: (184915, 435) (184915,) Unique y: [0 1]


1156/1156 [==============================] - 2s 2ms/step


1156/1156 [==============================] - 1s 885us/step


1156/1156 [==============================] - 1s 962us/step


1156/1156 [==============================] - 1s 855us/step


1156/1156 [==============================] - 1s 995us/step
저장 완료
 - all_fold_results.csv : Fold별 전체 성능 기록
 - model_summary.csv : 모델별 평균 성능 요약


In [ ]:
result_df

,Fold,Algorithm,TP,TN,FP,FN,Precision,Recall,Specificity,F1-score,Accuracy,Balanced Accuracy,AUC
0,1,Logistic Regression,24552,6645,2561,3225,0.905543,0.883897,0.721812,0.894589,0.843550,0.802854,0.898215
1,1,Random Forest,26861,3852,5354,916,0.833804,0.967023,0.418423,0.895486,0.830463,0.692723,0.879274
2,1,XGBoost,24571,6642,2564,3206,0.905509,0.884581,0.721486,0.894923,0.843982,0.803033,0.899825
3,1,LightGBM,24570,6699,2507,3207,0.907412,0.884545,0.727678,0.895833,0.845497,0.806111,0.901797
4,1,CatBoost,24474,6666,2540,3303,0.905975,0.881089,0.724093,0.893358,0.842008,0.802591,0.897610
5,1,MLP,22902,5901,3305,4875,0.873889,0.824495,0.640995,0.848474,0.778817,0.732745,0.808859
6,1,CNN,24842,6350,2856,2935,0.896888,0.894337,0.689768,0.895611,0.843415,0.792052,0.896943
7,2,Logistic Regression,24523,6584,2621,3255,0.903441,0.882821,0.715263,0.893012,0.841116,0.799042,0.894759
8,2,Random Forest,26780,3749,5456,998,0.830748,0.964072,0.407279,0.892458,0.825487,0.685675,0.872417
9,2,XGBoost,24605,6560,2645,3173,0.902936,0.885773,0.712656,0.894272,0.842684,0.799215,0.896496


In [ ]:
summary

,Precision,Recall,Specificity,F1-score,Accuracy,Balanced Accuracy,AUC
Algorithm,,,,,,,
LightGBM,0.9065,0.8858,0.7243,0.8960,0.8456,0.8051,0.9001
XGBoost,0.9042,0.8873,0.7162,0.8957,0.8447,0.8017,0.8986
Logistic Regression,0.9040,0.8854,0.7164,0.8946,0.8433,0.8009,0.8968
CatBoost,0.9050,0.8819,0.7206,0.8933,0.8418,0.8013,0.8961
CNN,0.8978,0.8923,0.6933,0.8950,0.8428,0.7928,0.8944
Random Forest,0.8318,0.9661,0.4106,0.8939,0.8278,0.6884,0.8756
MLP,0.8758,0.8218,0.6482,0.8479,0.7786,0.7350,0.8102
KNN,0.7620,0.8998,0.4378,0.8252,0.7458,0.6688,0.7670


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# -------------------------------
# 0. 데이터 불러오기
X = pd.read_csv("X_mm.csv")
y = pd.read_csv("y.csv").squeeze()

print(" 데이터 로드 완료:", X.shape, y.shape, "Unique y:", y.unique())

# -------------------------------
# 1. 상관계수 기반 (절대값 ≥ 0.9 → leakage 의심)
corrs = X.corrwith(y).abs().sort_values(ascending=False)
suspect_corr = corrs[corrs >= 0.9]

print("\n 상관계수 0.9 이상 변수:")
print(suspect_corr)

# -------------------------------
# 2. 단일 변수 AUC 기반 탐지 (AUC ≥ 0.95 → leakage 의심)
suspect_auc = {}
for col in X.columns:
    try:
        auc = roc_auc_score(y, X[col])
        if auc >= 0.95 or auc <= 0.05:  # 역상관도 포함
            suspect_auc[col] = auc
    except Exception:
        continue

suspect_auc = pd.Series(suspect_auc).sort_values(ascending=False)

print("\n 단일 변수 AUC 0.95 이상 변수:")
print(suspect_auc)

 데이터 로드 완료: (184915, 459) (184915,) Unique y: [1 2]

 상관계수 0.9 이상 변수:
Series([], dtype: float64)

 단일 변수 AUC 0.95 이상 변수:
Series([], dtype: object)


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd

# -------------------------------
# 0. 데이터 불러오기
X = pd.read_csv("X_mm.csv")
y = pd.read_csv("y.csv").squeeze()

print("✅ 데이터 로드 완료:", X.shape, y.shape)

# -------------------------------
# 1. 의심 키워드 정의
leak_keywords = ["희망", "사유", "의사", "향후", "기부"]

# -------------------------------
# 2. 의심 변수 후보 추출
suspect_cols = [c for c in X.columns if any(k in c for k in leak_keywords)]

print("\n📌 현실 예측에서 제외 고려할 변수 후보:")
print(suspect_cols)

# -------------------------------
# 3. 상관계수 계산 (y와 직접)
corrs = X[suspect_cols].corrwith(y).sort_values(key=abs, ascending=False)

print("\n📊 의심 변수와 타깃(y)의 상관계수:")
print(corrs)



✅ 데이터 로드 완료: (184915, 459) (184915,)

📌 현실 예측에서 제외 고려할 변수 후보:
['향후희망여가활동_1순위코드', '향후희망여가활동_2순위코드', '향후희망여가활동_3순위코드', '향후추가대상공공시설_1순위코드_2.0', '향후추가대상공공시설_1순위코드_3.0', '향후추가대상공공시설_1순위코드_4.0', '향후추가대상공공시설_1순위코드_5.0', '향후추가대상공공시설_1순위코드_6.0', '향후추가대상공공시설_1순위코드_7.0', '향후추가대상공공시설_1순위코드_8.0', '향후추가대상공공시설_2순위코드_2.0', '향후추가대상공공시설_2순위코드_3.0', '향후추가대상공공시설_2순위코드_4.0', '향후추가대상공공시설_2순위코드_5.0', '향후추가대상공공시설_2순위코드_6.0', '향후추가대상공공시설_2순위코드_7.0', '향후추가대상공공시설_2순위코드_8.0', '향후추가대상공공시설_3순위코드_2.0', '향후추가대상공공시설_3순위코드_3.0', '향후추가대상공공시설_3순위코드_4.0', '향후추가대상공공시설_3순위코드_5.0', '향후추가대상공공시설_3순위코드_6.0', '향후추가대상공공시설_3순위코드_7.0', '향후추가대상공공시설_3순위코드_8.0', '향후추가대상복지서비스_1순위코드_2.0', '향후추가대상복지서비스_1순위코드_3.0', '향후추가대상복지서비스_1순위코드_4.0', '향후추가대상복지서비스_1순위코드_5.0', '향후추가대상복지서비스_1순위코드_6.0', '향후추가대상복지서비스_1순위코드_7.0', '향후추가대상복지서비스_1순위코드_8.0', '향후추가대상복지서비스_2순위코드_2.0', '향후추가대상복지서비스_2순위코드_3.0', '향후추가대상복지서비스_2순위코드_4.0', '향후추가대상복지서비스_2순위코드_5.0', '향후추가대상복지서비스_2순위코드_6.0', '향후추가대상복지서비스_2순위코드_7.0', '향후추가대상복지서비스_2순위코드_8.0', '향후추가대상복지서비스_3순위코드_2.0', '향후추